# Build Dataset Generation Config

This notebook processes official energy data from:

- EPE (Empresa de Pesquisa Energética)
- ANEEL (Agência Nacional de Energia Elétrica)

The goal is to generate the probabilistic configuration files used by the synthetic dataset generator.

Outputs:
- consumption_profiles.csv
- energy_source_distribution.csv
- company_size_distribution_by_usage.csv
- usage_distribution_by_state.csv
- seasonality_state_class_month.csv

Clona Projeto direto do github para facilitar

In [1]:
# Mapping Portuguese categories to standardized English labels
category_map = {
    'comercial': 'comercial',
    'industrial': 'industrial',
    'outros': 'outros',
    'residencial': 'residencial',
    'rural': 'rural',
    'Comercial': 'comercial',
    'Industrial': 'industrial',
    'Outros': 'outros',
    'Residencial': 'residencial',
    'Rural': 'rural'
}

# # !git clone https://github.com/carbon-footprint-analysis/carbon-footprint-analysis.git

importa dados oficiais da empresa de pesquisa energetica e faz EDA + ETL para criar o dataset sintetico ainda !

Abaixo e o data set de consumo por categoria e estado o consumo esta em MW mas abaixo sera convertido para kwh

In [2]:
import pandas as pd

epe = pd.read_csv('../data/raw/epe_industrial_consumption_by_state.csv',
                    encoding='latin-1',
                    sep=';',
                    decimal=',')

In [3]:
epe.head()

,Coluna1,DataExcel,SetorIndustrial,UF,Regiao,Consumo,DataVersao
0,20251201,01/12/2025,05 - EXTRAÇÃO DE CARVÃO MINERAL,AM,Norte,15.000,23/02/2026
1,20251201,01/12/2025,05 - EXTRAÇÃO DE CARVÃO MINERAL,DF,Centro-Oeste,1.077,23/02/2026
2,20251201,01/12/2025,05 - EXTRAÇÃO DE CARVÃO MINERAL,ES,Sudeste,23.000,23/02/2026
3,20251201,01/12/2025,05 - EXTRAÇÃO DE CARVÃO MINERAL,MA,Nordeste,39.586,23/02/2026
4,20251201,01/12/2025,05 - EXTRAÇÃO DE CARVÃO MINERAL,MG,Sudeste,20994.000,23/02/2026


In [4]:
epe['DataExcel'] = pd.to_datetime(epe['DataExcel'], dayfirst=True)

In [5]:
epe.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 135676 entries, 0 to 135675
Data columns (total 7 columns):
 #   Column           Non-Null Count   Dtype         
---  ------           --------------   -----         
 0   Coluna1          135676 non-null  int64         
 1   DataExcel        135676 non-null  datetime64[ns]
 2   SetorIndustrial  135676 non-null  object        
 3   UF               135676 non-null  object        
 4   Regiao           135676 non-null  object        
 5   Consumo          135676 non-null  float64       
 6   DataVersao       135676 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(1), object(4)
memory usage: 7.2+ MB


In [6]:
epe['SetorIndustrial'].unique()

array(['05 - EXTRAÇÃO DE CARVÃO MINERAL',
       '06 - EXTRAÇÃO DE PETRÓLEO E GÁS NATURAL',
       '07 - EXTRAÇÃO DE MINERAIS METÁLICOS',
       '08 - EXTRAÇÃO DE MINERAIS NÃO-METÁLICOS',
       '09 - ATIVIDADES DE APOIO À EXTRAÇÃO DE MINERAIS',
       '10 - FABRICAÇÃO DE PRODUTOS ALIMENTÍCIOS',
       '11 - FABRICAÇÃO DE BEBIDAS',
       '12 - FABRICAÇÃO DE PRODUTOS DO FUMO',
       '13 - FABRICAÇÃO DE PRODUTOS TÊXTEIS',
       '14 - CONFECÇÃO DE ARTIGOS DO VESTUÁRIO E ACESSÓRIOS',
       '15 - PREPARAÇÃO DE COUROS E FABRICAÇÃO DE ARTEFATOS DE COURO, ARTIGOS PARA VIAGEM E CALÇADOS',
       '16 - FABRICAÇÃO DE PRODUTOS DE MADEIRA',
       '17 - FABRICAÇÃO DE CELULOSE, PAPEL E PRODUTOS DE PAPEL',
       '18 - IMPRESSÃO E REPRODUÇÃO DE GRAVAÇÕES',
       '19 - FABRICAÇÃO DE COQUE, DE PRODUTOS DERIVADOS DO PETRÓLEO E DE BIOCOMBUSTÍVEIS',
       '20 - FABRICAÇÃO DE PRODUTOS QUÍMICOS',
       '21 - FABRICAÇÃO DE PRODUTOS FARMOQUÍMICOS E FARMACÊUTICOS',
       '22 - FABRICAÇÃO DE PRODUTOS DE

In [7]:
df = epe.copy()

In [8]:
df['year'] = df['DataExcel'].dt.year
df['month'] = df['DataExcel'].dt.month

In [9]:
df_2025 = df[df['year'] == 2025].copy()
df_2025.drop(columns=['DataVersao', 'DataExcel'], inplace=True)

In [10]:
df_2025.head(5)

,Coluna1,SetorIndustrial,UF,Regiao,Consumo,year,month
0,20251201,05 - EXTRAÇÃO DE CARVÃO MINERAL,AM,Norte,15.000,2025,12
1,20251201,05 - EXTRAÇÃO DE CARVÃO MINERAL,DF,Centro-Oeste,1.077,2025,12
2,20251201,05 - EXTRAÇÃO DE CARVÃO MINERAL,ES,Sudeste,23.000,2025,12
3,20251201,05 - EXTRAÇÃO DE CARVÃO MINERAL,MA,Nordeste,39.586,2025,12
4,20251201,05 - EXTRAÇÃO DE CARVÃO MINERAL,MG,Sudeste,20994.000,2025,12


In [11]:
df_grouped = df.groupby('SetorIndustrial')['Consumo'].mean().reset_index()

In [12]:
df_grouped = df_grouped.rename(columns={'Consumo':'Consumo_MWh'})

In [13]:
df_grouped.head()

,SetorIndustrial,Consumo_MWh
0,05 - EXTRAÇÃO DE CARVÃO MINERAL,658.978726
1,06 - EXTRAÇÃO DE PETRÓLEO E GÁS NATURAL,8514.621995
2,07 - EXTRAÇÃO DE MINERAIS METÁLICOS,43036.799370
3,08 - EXTRAÇÃO DE MINERAIS NÃO-METÁLICOS,11104.061692
4,09 - ATIVIDADES DE APOIO À EXTRAÇÃO DE MINERAIS,739.495885


In [14]:
df_grouped['Consumo_kWh'] = (df_grouped['Consumo_MWh']) * 1000

In [15]:
df_grouped.head()

,SetorIndustrial,Consumo_MWh,Consumo_kWh
0,05 - EXTRAÇÃO DE CARVÃO MINERAL,658.978726,6.589787e+05
1,06 - EXTRAÇÃO DE PETRÓLEO E GÁS NATURAL,8514.621995,8.514622e+06
2,07 - EXTRAÇÃO DE MINERAIS METÁLICOS,43036.799370,4.303680e+07
3,08 - EXTRAÇÃO DE MINERAIS NÃO-METÁLICOS,11104.061692,1.110406e+07
4,09 - ATIVIDADES DE APOIO À EXTRAÇÃO DE MINERAIS,739.495885,7.394959e+05


In [16]:
df_grouped.to_csv('../data/processed/media_consumo_indutria_2025.csv')

Abertura do dados oficial consumo mensal por categoria a escala de energia esta em MW mas sera convertido em kwh.

Tambem foi feito o EDA e ETL para criação do dataset sintetico !

In [17]:
import pandas as pd

epe_categoria = pd.read_csv('../data/raw/EPE - Dados_abertos_Consumo_Mensal.CSV',
                    encoding='latin-1',
                    sep=';',
                    decimal=',')

In [18]:
df_2025 = df[df['year'] == 2025].copy()
df_2025.drop(columns=['DataVersao', 'DataExcel'], inplace=True)

In [19]:
epe_categoria.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 59243 entries, 0 to 59242
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Data            59243 non-null  int64 
 1   DataExcel       59243 non-null  object
 2   UF              59243 non-null  object
 3   Regiao          59243 non-null  object
 4   Sistema         59243 non-null  object
 5   Classe          59243 non-null  object
 6   TipoConsumidor  59243 non-null  object
 7   Consumo         59243 non-null  object
 8   Consumidores    59243 non-null  object
 9   DataVersao      59243 non-null  object
dtypes: int64(1), object(9)
memory usage: 4.5+ MB


In [20]:
epe_categoria['DataExcel'] = pd.to_datetime(epe_categoria['DataExcel'], dayfirst=True)

In [21]:
df = epe_categoria.copy()

In [22]:
df['year'] = df['DataExcel'].dt.year
df['month'] = df['DataExcel'].dt.month

In [23]:
df_2025 = df[df['year'] == 2025]

In [24]:
df_2025['Consumo'] = (
    df_2025['Consumo']
    .str.replace('.', '', regex=False)
    .str.replace(',', '.', regex=False)
    .astype(float)
)

df_2025['Consumidores'] = (
    df_2025['Consumidores']
    .astype(str)
    .str.replace('.', '', regex=False)
    .astype(float)
)

C:\Users\anlur\AppData\Local\Temp\ipykernel_28648\3063530973.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2025['Consumo'] = (
C:\Users\anlur\AppData\Local\Temp\ipykernel_28648\3063530973.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2025['Consumidores'] = (


In [25]:
df_2025.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3328 entries, 0 to 3327
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Data            3328 non-null   int64         
 1   DataExcel       3328 non-null   datetime64[ns]
 2   UF              3328 non-null   object        
 3   Regiao          3328 non-null   object        
 4   Sistema         3328 non-null   object        
 5   Classe          3328 non-null   object        
 6   TipoConsumidor  3328 non-null   object        
 7   Consumo         3328 non-null   float64       
 8   Consumidores    3328 non-null   float64       
 9   DataVersao      3328 non-null   object        
 10  year            3328 non-null   int32         
 11  month           3328 non-null   int32         
dtypes: datetime64[ns](1), float64(2), int32(2), int64(1), object(6)
memory usage: 312.0+ KB


In [26]:
df_2025.head()

,Data,DataExcel,UF,Regiao,Sistema,Classe,TipoConsumidor,Consumo,Consumidores,DataVersao,year,month
0,20251201,2025-12-01,AC,Norte,SISTEMAS ISOLADOS,Comercial,Cativo,175.7,478.0,23/02/2026,2025,12
1,20251201,2025-12-01,AC,Norte,SISTEMAS ISOLADOS,Industrial,Cativo,7.0,4.0,23/02/2026,2025,12
2,20251201,2025-12-01,AC,Norte,SISTEMAS ISOLADOS,Outros,Cativo,495.1,382.0,23/02/2026,2025,12
3,20251201,2025-12-01,AC,Norte,SISTEMAS ISOLADOS,Residencial,Cativo,1266.8,8862.0,23/02/2026,2025,12
4,20251201,2025-12-01,AC,Norte,SISTEMAS ISOLADOS,Rural,Cativo,1181.1,125.0,23/02/2026,2025,12


In [27]:
df_2025.drop(columns=['Data', 'DataVersao'], inplace=True)

C:\Users\anlur\AppData\Local\Temp\ipykernel_28648\3578452241.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2025.drop(columns=['Data', 'DataVersao'], inplace=True)


In [28]:
df_2025 = df_2025.rename(columns={'Consumo':'Consumo_MWh'})

In [29]:
df_simplificado = (
    df_2025
    .groupby('Classe')
    .agg({
        'Consumo_MWh': 'sum',
        'Consumidores': 'sum'
    })
)

df_simplificado['consumo_medio_MWh'] = df_simplificado['Consumo_MWh'] / df_simplificado['Consumidores']

df_simplificado = df_simplificado.sort_values('Consumo_MWh', ascending=False)

In [30]:
df_simplificado['Consumo_kWh'] = (df_simplificado['Consumo_MWh']) * 1000
df_simplificado['consumo_medio_KWh'] = (df_simplificado['consumo_medio_MWh']) * 1000

In [31]:
df_simplificado = df_simplificado[[
    'Consumidores',
    'Consumo_MWh',
    'Consumo_kWh',
    'consumo_medio_MWh',
    'consumo_medio_KWh'
]]

In [32]:
df_simplificado

,Consumidores,Consumo_MWh,Consumo_kWh,consumo_medio_MWh,consumo_medio_KWh
Classe,,,,,
Industrial,5.408760e+06,198858357.9,1.988584e+11,36.765979,36765.979245
Residencial,1.004322e+09,179155039.1,1.791550e+11,0.178384,178.384142
Comercial,7.399134e+07,102288641.3,1.022886e+11,1.382441,1382.440719
Outros,1.059433e+07,51581431.9,5.158143e+10,4.868779,4868.778995
Rural,4.604696e+07,30907605.1,3.090761e+10,0.671219,671.219246


In [33]:
df_simplificado.to_csv('../data/processed/consumo_medio_categoria_2025.csv')

Verificando a sanidade dos dados

In [34]:
df_grouped.head()

,SetorIndustrial,Consumo_MWh,Consumo_kWh
0,05 - EXTRAÇÃO DE CARVÃO MINERAL,658.978726,6.589787e+05
1,06 - EXTRAÇÃO DE PETRÓLEO E GÁS NATURAL,8514.621995,8.514622e+06
2,07 - EXTRAÇÃO DE MINERAIS METÁLICOS,43036.799370,4.303680e+07
3,08 - EXTRAÇÃO DE MINERAIS NÃO-METÁLICOS,11104.061692,1.110406e+07
4,09 - ATIVIDADES DE APOIO À EXTRAÇÃO DE MINERAIS,739.495885,7.394959e+05


In [35]:
df_grouped.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37 entries, 0 to 36
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   SetorIndustrial  37 non-null     object 
 1   Consumo_MWh      37 non-null     float64
 2   Consumo_kWh      37 non-null     float64
dtypes: float64(2), object(1)
memory usage: 1020.0+ bytes


In [36]:
df_simplificado.head()

,Consumidores,Consumo_MWh,Consumo_kWh,consumo_medio_MWh,consumo_medio_KWh
Classe,,,,,
Industrial,5.408760e+06,198858357.9,1.988584e+11,36.765979,36765.979245
Residencial,1.004322e+09,179155039.1,1.791550e+11,0.178384,178.384142
Comercial,7.399134e+07,102288641.3,1.022886e+11,1.382441,1382.440719
Outros,1.059433e+07,51581431.9,5.158143e+10,4.868779,4868.778995
Rural,4.604696e+07,30907605.1,3.090761e+10,0.671219,671.219246


In [37]:
df_simplificado.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5 entries, Industrial to Rural
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Consumidores       5 non-null      float64
 1   Consumo_MWh        5 non-null      float64
 2   Consumo_kWh        5 non-null      float64
 3   consumo_medio_MWh  5 non-null      float64
 4   consumo_medio_KWh  5 non-null      float64
dtypes: float64(5)
memory usage: 240.0+ bytes


Criação do consumption_profile.csv

In [38]:
df_profiles = df_simplificado.copy()

In [39]:
df_profiles.head()

,Consumidores,Consumo_MWh,Consumo_kWh,consumo_medio_MWh,consumo_medio_KWh
Classe,,,,,
Industrial,5.408760e+06,198858357.9,1.988584e+11,36.765979,36765.979245
Residencial,1.004322e+09,179155039.1,1.791550e+11,0.178384,178.384142
Comercial,7.399134e+07,102288641.3,1.022886e+11,1.382441,1382.440719
Outros,1.059433e+07,51581431.9,5.158143e+10,4.868779,4868.778995
Rural,4.604696e+07,30907605.1,3.090761e+10,0.671219,671.219246


In [40]:
def get_sigma(classe):
    if classe in ['Industrial', 'industrial']:
        return 0.6
    elif classe in ['Comercial', 'comercial']:
        return 0.4
    elif classe in ['Residencial', 'residencial']:
        return 0.3
    else:
        return 0.35

df_profiles['sigma'] = df_profiles.index.map(get_sigma)

In [41]:
map_usage = {
    'Industrial': 'industrial',
    'Comercial':  'comercial',
    'Residencial': 'residencial',
    'Rural':      'rural',
    'Outros':     'outros'
}

df_profiles['usage_type'] = df_profiles.index.map(map_usage)

In [42]:
import numpy as np

df_profiles['mu'] = np.log(df_profiles['consumo_medio_KWh'])

In [43]:
df_final = df_profiles.copy()
df_final['fuel_type'] = 'electric'
df_final['distribution_type'] = 'lognormal'
df_final['param_1'] = df_profiles['mu']
df_final['param_2'] = df_profiles['sigma']
df_final['param_1_name'] = 'mu'
df_final['param_2_name'] = 'sigma'
df_final['unit'] = 'kWh'
df_final['is_energy_based'] = True

In [44]:
df_final.head()

,Consumidores,Consumo_MWh,Consumo_kWh,consumo_medio_MWh,consumo_medio_KWh,sigma,usage_type,mu,fuel_type,distribution_type,param_1,param_2,param_1_name,param_2_name,unit,is_energy_based
Classe,,,,,,,,,,,,,,,,
Industrial,5.408760e+06,198858357.9,1.988584e+11,36.765979,36765.979245,0.60,industrial,10.512328,electric,lognormal,10.512328,0.60,mu,sigma,kWh,True
Residencial,1.004322e+09,179155039.1,1.791550e+11,0.178384,178.384142,0.30,residential,5.183939,electric,lognormal,5.183939,0.30,mu,sigma,kWh,True
Comercial,7.399134e+07,102288641.3,1.022886e+11,1.382441,1382.440719,0.40,commercial,7.231606,electric,lognormal,7.231606,0.40,mu,sigma,kWh,True
Outros,1.059433e+07,51581431.9,5.158143e+10,4.868779,4868.778995,0.35,other,8.490598,electric,lognormal,8.490598,0.35,mu,sigma,kWh,True
Rural,4.604696e+07,30907605.1,3.090761e+10,0.671219,671.219246,0.35,agriculture,6.509096,electric,lognormal,6.509096,0.35,mu,sigma,kWh,True


In [45]:
df_export = df_final[[
    'usage_type',
    'fuel_type',
    'distribution_type',
    'param_1',
    'param_2',
    'param_1_name',
    'param_2_name',
    'unit',
    'is_energy_based'
]].copy()

In [46]:
df_export.head()

,usage_type,fuel_type,distribution_type,param_1,param_2,param_1_name,param_2_name,unit,is_energy_based
Classe,,,,,,,,,
Industrial,industrial,electric,lognormal,10.512328,0.60,mu,sigma,kWh,True
Residencial,residential,electric,lognormal,5.183939,0.30,mu,sigma,kWh,True
Comercial,commercial,electric,lognormal,7.231606,0.40,mu,sigma,kWh,True
Outros,other,electric,lognormal,8.490598,0.35,mu,sigma,kWh,True
Rural,agriculture,electric,lognormal,6.509096,0.35,mu,sigma,kWh,True


In [47]:
df_export.to_csv('../data/processed/v2_consumption_profiles.csv', index=False)

transformar config em dados sinteticos reais

carergando profile

In [48]:
import pandas as pd
import numpy as np

profiles = pd.read_csv('../data/processed/v2_consumption_profiles.csv')

FileNotFoundError: [Errno 2] No such file or directory: '/content/v2_consumption_profiles.csv'

gerar função

In [ ]:
def generate_consumption(profile_row):

    if profile_row['distribution_type'] == 'lognormal':
        mu = profile_row['param_1']
        sigma = profile_row['param_2']

        value = np.random.lognormal(mu, sigma)

        # LIMITADOR
        max_value = 10 * np.exp(mu)
        value = min(value, max_value)

        return value

    else:
        raise ValueError("Distribuição não suportada")

gerar evento

In [ ]:
usage_probs = {
    'residencial': 0.6,
    'comercial': 0.2,
    'industrial': 0.1,
    'rural': 0.05,
    'outros': 0.05
}

In [ ]:
def generate_event(profiles):

    weights = profiles['usage_type'].map(usage_probs)

    row = profiles.sample(1, weights=weights).iloc[0]

    consumption = generate_consumption(row)

    return {
        'usage_type': row['usage_type'],
        'fuel_type': row['fuel_type'],
        'energy_kwh': consumption
    }

gerar dataset

In [ ]:
def generate_dataset(n, profiles):
    data = []

    for _ in range(n):
        event = generate_event(profiles)
        data.append(event)

    return pd.DataFrame(data)

test

In [ ]:
df = generate_dataset(10000, profiles)

In [ ]:
df.head()

In [ ]:
df.to_csv('df_gerado_teste.csv')

In [ ]:
df['usage_type'].value_counts(normalize=True)

In [ ]:
df.groupby('usage_type')['energy_kwh'].describe()

In [ ]:
df['energy_kwh'].max()

In [ ]:
(df['energy_kwh'] < 0).sum()

In [ ]:
import matplotlib.pyplot as plt

df[df['usage_type']=='industrial']['energy_kwh'].hist(bins=50)
plt.title('Industrial')
plt.show()

criando o energy_source_distribution.csv

In [ ]:
df_aneel = pd.read_csv('../data/raw/aneel_generation.csv',
                       encoding='latin-1',
                       sep=';',
                       decimal=',')

In [ ]:
df_aneel.info()

In [ ]:
df_aneel.head()

In [ ]:
df_aneel['AnoReferencia'].unique()

In [ ]:
df_aneel = df_aneel[df_aneel['AnoReferencia']==2025]

In [ ]:
df_aneel['SigTipoGeracao'].unique()

In [ ]:
df_grouped = df_aneel.groupby('SigTipoGeracao')['MdaPotenciaInstaladaKW'].sum()

In [ ]:
df_grouped.head()

converter para probabilidade

In [ ]:
df_dist = df_grouped / df_grouped.sum()

transformar e dataframe

In [ ]:
df_dist.info()

In [ ]:
df_grouped = df_aneel.groupby('SigTipoGeracao')['MdaPotenciaInstaladaKW'].sum().reset_index()

df_grouped.columns = ['energy_source', 'total_generation']

In [ ]:
df_grouped['probability'] = df_grouped['total_generation'] / df_grouped['total_generation'].sum()

df_dist = df_grouped[['energy_source', 'probability']]
df_dist = df_dist.copy()

In [ ]:
df_dist.head()

In [ ]:
df_dist['probability'].sum()

In [ ]:
map_sources = {
    'UHE': 'hidrelétrica',      # Hidrelétrica grande
    'PCH': 'hidrelétrica',      # Pequena central hidrelétrica
    'CGH': 'hidrelétrica',      # Central geradora hidráulica

    'EOL': 'eólica',       # Eólica

    'UFV': 'solar',      # Solar fotovoltaica

    # se aparecer depois:
    'UTE': 'térmica',    # termoelétrica (genérico)
    'UTN': 'nuclear',    # nuclear
}
df_dist['energy_source'] = df_dist['energy_source'].map(map_sources)

In [ ]:
df_dist = df_dist.groupby('energy_source')['probability'].sum().reset_index()

In [ ]:
df_dist['probability'] = df_dist['probability'] / df_dist['probability'].sum()

In [ ]:
df_dist = df_dist.sort_values('probability', ascending=False)

In [ ]:
df_dist

In [ ]:
df_dist.to_csv('../data/processed/v2_energy_source_distribution.csv')

Inserir a emissão co2

In [ ]:
emission_factors = pd.read_csv('../data/processed/v2_energy_source_emission_factors.csv')

In [ ]:
def calculate_emission(energy_kwh, energy_source, emission_df):

    row = emission_df.loc[
        emission_df['energy_source'] == energy_source
    ]

    if row.empty:
        raise ValueError(f"Fonte não encontrada: {energy_source}")

    factor = row['emission_factor'].values[0]

    return energy_kwh * factor

In [ ]:
def sample_energy_source(dist_df):
    return dist_df.sample(1, weights=dist_df['probability']).iloc[0]['energy_source']

In [ ]:
def normalize_usage_types(series):

    mapping = {
        'residencial': 'residencial',
        'rural': 'rural',
        'outros': 'outros',
        'industrial': 'industrial',
        'comercial': 'comercial'
    }

    return (
        series
        .str.strip()
        .str.lower()
        .replace(mapping)
    )

In [ ]:
def generate_event(profiles, energy_dist, emission_df):

    usage_types = normalize_usage_types(profiles['usage_type'])

    weights = usage_types.map(usage_probs)

    if weights.sum() == 0:
        raise ValueError("usage_probs não corresponde aos usage_type do profile")

    row = profiles.sample(1, weights=weights).iloc[0]

    consumption = generate_consumption(row)

    energy_source = sample_energy_source(energy_dist)

    co2 = calculate_emission(consumption, energy_source, emission_df)

    return {
        'usage_type': row['usage_type'],
        'fuel_type': row['fuel_type'],
        'energy_kwh': consumption,
        'energy_source': energy_source,
        'co2_emission': co2
    }

In [ ]:
import pandas as pd
profiles = pd.read_csv('../data/processed/v2_consumption_profiles.csv')

energy_dist = pd.read_csv('../data/processed/v2_energy_source_distribution.csv')

emission_df = pd.read_csv('../data/processed/v2_energy_source_emission_factors.csv')

In [ ]:
event = generate_event(profiles, energy_dist, emission_df)
event

In [ ]:
def generate_dataset(n, profiles, energy_dist, emission_df):
    data = []

    for _ in range(n):
        event = generate_event(profiles, energy_dist, emission_df)
        data.append(event)

    return pd.DataFrame(data)

In [ ]:
df = generate_dataset(10000, profiles, energy_dist, emission_df)
df.describe()

In [ ]:
df.head()

Adicionando ID e datas

In [ ]:
company_id = f"C{np.random.randint(1000,9999)}"

In [ ]:
date = pd.Timestamp('2025-01-01') + pd.to_timedelta(np.random.randint(0, 365), unit='D')

In [ ]:
states = [
'SP','MG','RJ','BA','PR','RS','SC','GO','PE','CE',
'PA','MT','ES','DF','MS','MA','RN','PB','AL','PI',
'RO','SE','TO','AC','AP','RR','AM'
]

state_weights = [
0.22,0.10,0.09,0.07,0.06,0.06,0.04,0.04,0.04,0.04,
0.03,0.03,0.02,0.02,0.02,0.02,0.015,0.015,0.01,0.01,
0.008,0.008,0.006,0.005,0.005,0.003,0.02
]


In [ ]:
len(states), len(state_weights)

In [ ]:
state_weights = np.array(state_weights)
state_weights = state_weights / state_weights.sum()
state_weights

In [ ]:
def generate_event(profiles, energy_dist, emission_df):

    usage_types = normalize_usage_types(profiles['usage_type'])

    weights = usage_types.map(usage_probs)

    if weights.sum() == 0:
        raise ValueError("usage_probs não corresponde aos usage_type do profile")

    row = profiles.sample(1, weights=weights).iloc[0]

    consumption = generate_consumption(row)

    energy_source = sample_energy_source(energy_dist)

    co2 = calculate_emission(consumption, energy_source, emission_df)

    return {
        'company_id': f"C{np.random.randint(100000,999999)}",
        'date': pd.Timestamp('2025-01-01') + pd.to_timedelta(np.random.randint(0,365), unit='D'),
        'state': np.random.choice(states, p=state_weights),
        'usage_type': row['usage_type'],
        'fuel_type': row['fuel_type'],
        'energy_kwh': consumption,
        'energy_source': energy_source,
        'co2_emission': co2
    }

In [ ]:
def generate_dataset(n, profiles, energy_dist, emission_df):
    data = []

    for _ in range(n):
        event = generate_event(profiles, energy_dist, emission_df)
        data.append(event)

    return pd.DataFrame(data)

In [ ]:
df = generate_dataset(10000, profiles, energy_dist, emission_df)
df.describe()

In [ ]:
df

Adicionando regras mais realistas, do jeito que esta esta pode criar industria pesadas na msm probabilidade entre ACRE e São Paulo

In [ ]:
df_epe = pd.read_csv('../data/raw/EPE - Dados_abertos_Consumo_Mensal.CSV',
                    encoding='latin-1',
                    sep=';',
                    decimal=',')

In [ ]:
df_epe.columns

In [ ]:
df_epe.head(5)

In [ ]:
df_epe.info()

In [ ]:
df_epe['Consumo'] = (
    df_epe['Consumo']
    .str.replace('.', '', regex=False)
    .str.replace(',', '.', regex=False)
    .astype(float)
)

In [ ]:
df_sector_state = (
    df_epe
    .groupby(['UF','Classe'])['Consumo']
    .sum()
    .reset_index()
)

In [ ]:
df_sector_state.head(20)

In [ ]:
df_sector_state['probability'] = (
    df_sector_state
    .groupby('UF')['Consumo']
    .transform(lambda x: x / x.sum())
)

In [ ]:
usage_distribution_by_state = df_sector_state.pivot(
    index='UF',
    columns='Classe',
    values='probability'
)
usage_distribution_by_state.columns = [category_map.get(c, c.lower()) for c in usage_distribution_by_state.columns]

In [ ]:
usage_distribution_by_state.to_csv('../data/processed/v2_usage_distribution_by_state.csv')

distribuição por estado

In [ ]:
df_epe

In [ ]:
state_consumption = df_epe.groupby("UF")["Consumo"].sum()

state_distribution = state_consumption / state_consumption.sum()


In [ ]:
state_distribution

In [ ]:
state_distribution.to_csv("../data/processed/v2_state_distribution.csv")

In [ ]:
df_state = pd.read_csv("../data/processed/v2_state_distribution.csv")

states = df_state["UF"]
state_weights = df_state["Consumo"]


In [ ]:
def generate_event(profiles, energy_dist, emission_df):

    # sorteia estado
    state = np.random.choice(states, p=state_weights)

    # pega probabilidades de setor para esse estado
    usage_probs_state = usage_distribution_by_state.loc[state].values
    usage_probs_state = usage_probs_state / usage_probs_state.sum()

    # sorteia setor
    usage_types = profiles['usage_type'].unique()

    usage_type = np.random.choice(
        usage_types,
        p=usage_probs_state
    )

    # filtra profiles para esse setor
    profiles_subset = profiles[profiles['usage_type'] == usage_type]

    # escolhe linha de profile
    row = profiles_subset.sample(1).iloc[0]

    # gera consumo
    consumption = generate_consumption(row)

    # sorteia fonte de energia
    energy_source = sample_energy_source(energy_dist)

    # calcula CO2
    co2 = calculate_emission(consumption, energy_source, emission_df)

    return {
        'company_id': f"C{np.random.randint(100000,999999)}",
        'date': pd.Timestamp('2025-01-01') + pd.to_timedelta(np.random.randint(0,365), unit='D'),
        'state': np.random.choice(states, p=state_weights),
        'usage_type': usage_type,
        'fuel_type': row['fuel_type'],
        'energy_kwh': consumption,
        'energy_source': energy_source,
        'co2_emission': co2
    }

In [ ]:
def generate_dataset(n, profiles, energy_dist, emission_df):
    data = []

    for _ in range(n):
        event = generate_event(profiles, energy_dist, emission_df)
        data.append(event)

    return pd.DataFrame(data)

In [ ]:
df = generate_dataset(10000, profiles, energy_dist, emission_df)
df.describe()

In [ ]:
df

In [ ]:
df = generate_dataset(10000, profiles, energy_dist, emission_df)

In [ ]:
df.info()

Extraindo a probabilidade do tramanho do negocio small, medium e large. Para criar dataset mais realista !

In [ ]:
df_epe = pd.read_csv('../data/raw/EPE - Dados_abertos_Consumo_Mensal.CSV',
                    encoding='latin-1',
                    sep=';',
                    decimal=',')

In [ ]:
df_epe['Consumo'] = (
    df_epe['Consumo']
    .str.replace('.', '', regex=False)
    .str.replace(',', '.', regex=False)
    .astype(float)
)

In [ ]:
df_epe['DataExcel'] = pd.to_datetime(df_epe['DataExcel'], dayfirst=True)

In [ ]:
df_epe['year'] = df_epe['DataExcel'].dt.year
df_epe['month'] = df_epe['DataExcel'].dt.month

In [ ]:
df_2025 = df_epe[df_epe['year'] == 2025].copy()

In [ ]:
df_2025

In [ ]:
df_2025['DataExcel'].unique()

In [ ]:
df_2025['year'].unique()

In [ ]:
df_epe = df_2025.copy()

In [ ]:
def classify_company_size(row):

    usage = row['Classe']
    energy = row['Consumo']

    if usage == 'Industrial':
        if energy < 20000:
            return 'small'
        elif energy < 100000:
            return 'medium'
        else:
            return 'large'

    if usage == 'Comercial':
        if energy < 5000:
            return 'small'
        elif energy < 30000:
            return 'medium'
        else:
            return 'large'

    if usage == 'Residencial':
        return 'small'

    if usage == 'Rural':
        if energy < 10000:
            return 'small'
        elif energy < 50000:
            return 'medium'
        else:
            return 'large'

    if usage == 'Outros':
        if energy < 10000:
            return 'small'
        elif energy < 50000:
            return 'medium'
        else:
            return 'large'

In [ ]:
df_epe['company_size'] = df_epe.apply(classify_company_size, axis=1)

In [ ]:
df_epe['company_size'].value_counts()

In [ ]:
size_dist = (
    df_epe
    .groupby('Classe')['company_size']
    .value_counts(normalize=True)
    .reset_index(name='probability')
)

In [ ]:
size_dist

In [ ]:
size_dist.to_csv(
    '../data/processed/v2_company_size_distribution_by_usage.csv',
    index=False
)

calculando a variancia por mes, meses podem impactar no consumo de energia !

In [ ]:
df_epe = pd.read_csv('../data/raw/EPE - Dados_abertos_Consumo_Mensal.CSV',
                    encoding='latin-1',
                    sep=';',
                    decimal=',')

In [ ]:
df_epe['Consumo'] = (
    df_epe['Consumo']
    .str.replace('.', '', regex=False)
    .str.replace(',', '.', regex=False)
    .astype(float)
)

In [ ]:
df_epe['DataExcel'] = pd.to_datetime(df_epe['DataExcel'], dayfirst=True)

In [ ]:
df_epe['year'] = df_epe['DataExcel'].dt.year
df_epe['month'] = df_epe['DataExcel'].dt.month

In [ ]:
df_2025 = df_epe[df_epe['year'] == 2025].copy()

In [ ]:
df_2025

In [ ]:
df_epe = df_2025.copy()

In [ ]:
df_epe['month'] = df_epe['DataExcel'].dt.month

In [ ]:
monthly_stats = (
    df_epe
    .groupby(['UF','Classe','month'])['Consumo']
    .agg(['mean','var'])
    .reset_index()
)

In [ ]:
annual_mean = (
    df_epe
    .groupby(['UF','Classe'])['Consumo']
    .mean()
    .reset_index()
    .rename(columns={'Consumo':'annual_mean'})
)

In [ ]:
monthly_stats = monthly_stats.merge(
    annual_mean,
    on=['UF','Classe']
)

In [ ]:
monthly_stats['seasonal_factor'] = (
    monthly_stats['mean'] /
    monthly_stats['annual_mean']
)

In [ ]:
monthly_stats

In [ ]:
monthly_stats = monthly_stats[[
    'UF',
    'Classe',
    'month',
    'seasonal_factor',
    'var'
]]

monthly_stats.to_csv(
    '../data/processed/v2_seasonality_state_class_month.csv',
    index=False
)

Gerando df completo

In [ ]:
# !ls

In [ ]:
from pathlib import Path

ROOT = Path().resolve().parent

In [ ]:
# !pwd

In [ ]:
# !git pull

import csvs

In [ ]:
import pandas as pd
import numpy as np

profiles = pd.read_csv('../data/processed/v2_consumption_profiles.csv')

energy_dist = pd.read_csv('../data/processed/v2_energy_source_distribution.csv')

emission_df = pd.read_csv('../data/processed/v2_energy_source_emission_factors.csv')

usage_distribution_by_state = pd.read_csv(
    '../data/processed/v2_usage_distribution_by_state.csv',
    index_col=0
)

company_size_dist = pd.read_csv(
    '../data/processed/v2_company_size_distribution_by_usage.csv'
)

seasonality = pd.read_csv(
    '../data/processed/v2_seasonality_state_class_month.csv'
)

state_dist = pd.read_csv('../data/processed/v2_state_distribution.csv')
states = state_dist['UF'].values
state_weights = state_dist['Consumo'].values / state_dist['Consumo'].sum()


Sanity para integrar todos as tabelas estava dando MUITOS erros !

In [ ]:
# Todos os CSVs já estão em português minúsculo após a conversão.
# Verificação de consistência:

assert set(profiles['usage_type'].unique()) <= {'comercial', 'industrial', 'outros', 'residencial', 'rural'}, \
    f"usage_type inesperado em profiles: {set(profiles['usage_type'].unique())}"

assert set(company_size_dist['Classe'].unique()) <= {'comercial', 'industrial', 'outros', 'residencial', 'rural'}, \
    f"Classe inesperada em company_size_dist: {set(company_size_dist['Classe'].unique())}"

assert set(seasonality['Classe'].unique()) <= {'comercial', 'industrial', 'outros', 'residencial', 'rural'}, \
    f"Classe inesperada em seasonality: {set(seasonality['Classe'].unique())}"

setores_state = set(usage_distribution_by_state.columns)
setores_profiles = set(profiles['usage_type'].unique())
assert setores_state == setores_profiles, \
    f"Mismatch setores: state={setores_state} | profiles={setores_profiles}"

print("✓ Todas as categorias estão consistentes em PT-BR")
print("  Setores:", sorted(setores_profiles))
print("  Fontes:", sorted(energy_dist['energy_source'].unique()))
print("  Portes:", sorted(company_size_dist['company_size'].unique()))


In [ ]:
print('profiles: '+profiles.columns,'\n')
print('energy_dist: '+energy_dist.columns,'\n')
print('emission_df: '+emission_df.columns,'\n')
print('usage_distribution_by_state: '+usage_distribution_by_state.columns,'\n')
print('company_size_dist: '+company_size_dist.columns,'\n')
print('seasonality: '+seasonality.columns,'\n')

In [ ]:
def show_uniques(df, name):
    print(f"\n{name}")
    for col in df.select_dtypes(include='object').columns:
        print(col, "->", df[col].unique())

In [ ]:
show_uniques(profiles, "profiles")
show_uniques(company_size_dist, "company_size_dist")
show_uniques(usage_distribution_by_state, "usage_distribution")
show_uniques(seasonality, "seasonality")

In [ ]:
print(set(profiles['usage_type']))
print(set(company_size_dist['Classe']))
print(set(usage_distribution_by_state.columns))
print(set(seasonality['Classe']))

In [ ]:
print("usage_types possíveis:", usage_distribution_by_state.columns.tolist())
print("classes em company_size:", company_size_dist['Classe'].unique())

In [ ]:
for u in usage_distribution_by_state.columns:
    subset = company_size_dist[company_size_dist['Classe'] == u]
    print(u, "->", len(subset))

In [ ]:
print(energy_dist['probability'].sum())

In [ ]:
print(company_size_dist.groupby('Classe')['probability'].sum())

função geração consumo

In [ ]:
def generate_consumption(profile_row):

    if profile_row['distribution_type'] == 'lognormal':

        mu = profile_row['param_1']
        sigma = profile_row['param_2']

        value = np.random.lognormal(mu, sigma)

        max_value = 10 * np.exp(mu)
        value = min(value, max_value)

        return value

    else:
        raise ValueError("Distribuição não suportada")

fonte energetica

In [ ]:
def sample_energy_source(dist_df):

    return dist_df.sample(
        1,
        weights=dist_df['probability']
    ).iloc[0]['energy_source']

calcular emissão

In [ ]:
def calculate_emission(energy_kwh, energy_source, emission_df):

    row = emission_df.loc[
        emission_df['energy_source'] == energy_source
    ]

    factor = row['emission_factor'].values[0]

    return energy_kwh * factor

amostra tamanho da empresa

In [ ]:
def sample_company_size(setor):
    """
    Sorteia o porte da empresa para o setor dado.
    Retorna 'pequena' como fallback se o setor não for encontrado.
    """
    subset = company_size_dist[company_size_dist['Classe'] == setor].copy()

    if subset.empty:
        return 'pequena'

    portes = subset['company_size'].values
    probs  = subset['probability'].values.astype(float)

    total = probs.sum()
    if total <= 0:
        return 'pequena'

    probs = probs / total

    return np.random.choice(portes, p=probs)


aplicar sazionalidade

In [ ]:
def apply_seasonality(consumo, estado, setor, mes):
    """
    Ajusta o consumo pelo fator de sazonalidade (estado × setor × mês).
    Retorna o consumo original se não houver dado para a combinação.
    """
    subset = seasonality[
        (seasonality['UF']     == estado) &
        (seasonality['Classe'] == setor)  &
        (seasonality['month']  == mes)
    ]

    if subset.empty:
        return consumo

    fator = subset['seasonal_factor'].values[0]
    return consumo * fator


gerar evento (juntar tudo)

In [ ]:
def generate_event(profiles):

    # -------------------------------------------------
    # 1) Escolha do estado
    # -------------------------------------------------
    estado = np.random.choice(states, p=state_weights)


    # -------------------------------------------------
    # 2) Sorteio do mês
    # -------------------------------------------------
    mes = np.random.randint(1, 13)


    # -------------------------------------------------
    # 3) Setor econômico condicionado ao estado
    # -------------------------------------------------
    probs_estado = usage_distribution_by_state.loc[estado].values.astype(float)
    probs_estado = probs_estado / probs_estado.sum()

    setores = usage_distribution_by_state.columns.tolist()

    setor = np.random.choice(setores, p=probs_estado)


    # -------------------------------------------------
    # 4) Perfil do setor
    # -------------------------------------------------
    profiles_subset = profiles[profiles['usage_type'] == setor]

    if profiles_subset.empty:
        raise ValueError(f"Nenhum perfil encontrado para o setor: {setor!r}")

    row = profiles_subset.sample(1).iloc[0]


    # -------------------------------------------------
    # 5) Consumo base
    # -------------------------------------------------
    consumo = generate_consumption(row)


    # -------------------------------------------------
    # 6) Ajuste de sazonalidade
    # -------------------------------------------------
    consumo = apply_seasonality(consumo, estado, setor, mes)


    # -------------------------------------------------
    # 7) Ruído operacional (~8%)
    # -------------------------------------------------
    consumo = consumo * np.random.normal(1, 0.08)


    # -------------------------------------------------
    # 8) Porte da empresa
    # -------------------------------------------------
    porte = sample_company_size(setor)


    # -------------------------------------------------
    # 9) Fonte de energia
    # -------------------------------------------------
    fonte = sample_energy_source(energy_dist)


    # -------------------------------------------------
    # 10) Emissão de CO₂ base
    # -------------------------------------------------
    co2 = calculate_emission(consumo, fonte, emission_df)


    # -------------------------------------------------
    # 11) Variação do fator de emissão (~6%)
    # -------------------------------------------------
    co2 = co2 * np.random.normal(1, 0.06)


    # -------------------------------------------------
    # 12) Ruído de medição (~3%)
    # -------------------------------------------------
    co2 = co2 * np.random.normal(1, 0.03)


    # -------------------------------------------------
    # 13) Data dentro do mês sorteado
    # -------------------------------------------------
    data = (
        pd.Timestamp('2025-01-01')
        + pd.DateOffset(months=mes - 1)
        + pd.to_timedelta(np.random.randint(0, 28), unit='D')
    )


    # -------------------------------------------------
    # 14) Retorno do evento
    # -------------------------------------------------
    return {
        'id_empresa':      f"C{np.random.randint(100000, 999999)}",
        'data':            data,
        'estado':          estado,
        'setor':           setor,
        'porte':           porte,
        'tipo_combustivel': row['fuel_type'],
        'consumo_kwh':     consumo,
        'fonte_energia':   fonte,
        'emissao_co2':     co2,
    }


gerar dataset

In [ ]:
def generate_dataset(n):
    """Gera n eventos sintéticos de consumo e emissão de CO₂."""
    data = []

    for _ in range(n):
        data.append(generate_event(profiles))

    return pd.DataFrame(data)


Montar dataset com tudo

In [ ]:
df = generate_dataset(100000)

df.head()

In [ ]:
df.to_csv('../data/processed/synthetic_energy_emissions_dataset.csv')